# Trabajo Práctico  - Calibración y Pose de la Cámara

## Resolución

In [ ]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import os
from matplotlib import pyplot as plt

import importlib
import ALC
import MODULO_ALC

from MODULO_ALC import rq, svd_reducida, calculaQR, multiplicacionMatricial, inversa, traspuesta, QR_con_HH, calculaCholesky, norma, productoExterno
from ALC import determinante3x3

importlib.reload(ALC)
print(dir(ALC))



ImportError: cannot import name 'determinante3x3' from 'ALC' (consider renaming 'j:\\Downloads\\TP\\template-alumnos-20261025\\Git\\TP-ALC\\TP ALC\\ALC.py' if it has the same name as a library you intended to import)

## GRUPO: A=TR

# Franco Nahuel Chazarreta Villagra
* 
* integrante

### Ejercicio 1 - DLT ecuaciones formales

Siendo $Q\in \mathbb{R}^{2n\times 12}$, ¿qué rango debe tener para asegurar una solución no nula única a menos de un factor de escala? ¿Cuántos puntos, como mínimo, aseguran la solución y qué condición deben cumplir?

Encontrar el vector $\mathbf{m}$ que resuelve la minimización de la ecuación 6.

### Ejercicio 2 - Calibración con puntos 3D

a) Utilizando los datos de la carpeta $\texttt{calib/3d}$ calcular $\mathbf{R}$,  $\mathbf{t}$ y $\mathbf{K}$.

Calcular un nuevo set de parámetros para 

b) $\alpha_u = \alpha_v$ 

c) $K_{12}=0$



In [16]:
imgPts_todas = np.loadtxt('calib/3d/detected_corners.txt')
objPts = np.loadtxt('calib/3d/p_W_corners.txt', delimiter=',')

imgPts = imgPts_todas[0].reshape(12, 2)

Q = ALC.obtenerQ3d(imgPts, objPts)

M = ALC.resolverDLT3D(Q)

R, t, K = ALC.devolverMatricesCalibracion3d(M)

print("Q:", Q.shape)
print("M:", M.shape)
print("K:", K.shape)
print("R:", R.shape)
print("t:", t.shape)

print("K:")
print(K)

print("R:")
print(R)

print("t:")
print(t)

print("R * R^T:")
print(multiplicacionMatricial(R, traspuesta(R)))

print("Determinante de R:")
print(determinante3x3(R))


Q: (24, 12)
M: (3, 4)
K: (3, 3)
R: (3, 3)
t: (3,)
K:
[[ 1.68213860e+00  1.41089656e+00  2.62774537e-01]
 [-7.37093010e-18  6.73523345e-02  6.61182432e-02]
 [-1.93616631e-16  8.37868512e-17  1.00000000e+00]]
R:
[[-0.1808389  -0.16305747 -0.96990183]
 [-0.94565723 -0.24214026  0.21702649]
 [-0.27024007  0.95644151 -0.11040809]]
t:
[-32.46766863   0.25183293  -0.27816146]
R * R^T:
[[ 1.00000000e+00  5.55111512e-17 -2.77555756e-16]
 [ 5.55111512e-17  1.00000000e+00  1.00613962e-16]
 [-2.77555756e-16  1.00613962e-16  1.00000000e+00]]
Determinante de R:
0.9999999999999998


### Ejercicio 3 - Calibración con puntos coplanares

Resolver la calibración para el caso de puntos coplanares, trabajando sobre los datos de la carpeta $\texttt{calib/coplanar}$ con las imágenes $\texttt{Im\_R\_1.png}$ a $\texttt{Im\_R\_20.png}$.


a) Obtener la matriz $\mathbf{K}$

b) Encontrar la pose de $\texttt{Im\_R\_1.png}$, $\texttt{Im\_R\_10.png}$ y $\texttt{Im\_R\_20.png}$.

### Ejercicio 4 - Fútbol

a) Calcular la homografía de la cancha de futbol de la foto de $\texttt{gol-enzo.png}$. Encontrar la distancia a la que se encontraba Enzo. 
    Dibujar sobre la fotografía círculos de 1 metro de diámetro alrededor de los jugadores argentinos.

In [ ]:
# a) Distancia de Enzo a la línea del arco

img = cv2.imread(r'path de imagen, gol de enzo')
img_rgb = img[:, :, ::-1]

# Mostramos la imagen con grilla para leer a ojo las coordenadas (u, v)
# (la grilla y los ejes son solo para el desarrollo: no están en la lista de
#  funciones permitidas, hay que sacarlos de la versión final de la entrega)
plt.figure(figsize=(12, 8))
plt.imshow(img_rgb)
plt.grid(True, color='red', linestyle='--', linewidth=0.5)
plt.xlabel('u (píxeles horizontal)')
plt.ylabel('v (píxeles vertical)')
plt.title('Identificación de píxeles (u, v)')
plt.show()

NameError: name 'cv2' is not defined

In [ ]:
# Puntos de referencia en la imagen (u, v), leídos del gráfico anterior
imgPts = np.array([
    [160, 400],  # 1. Esquina frontal izq del área grande
    [980, 400],  # 2. Esquina frontal der del área grande
    [280,  98],  # 3. Línea de meta / área grande (izq)
    [745,  98],  # 4. Línea de meta / área grande (der)
    [362, 175],  # 5. Esquina frontal izq del área chica
    [628, 175],  # 6. Esquina frontal der del área chica
    [478, 255],  # 7. Punto penal
], dtype=np.float32)

# Los mismos puntos en coordenadas de la cancha (X, Y) en metros,
# con el origen en la esquina del arco y X medido desde la línea de meta.
# Mismo orden que imgPts (el punto i de uno corresponde al punto i del otro)
objPts = np.array([
    [16.5, 13.84],  # 1. Esquina frontal izq del área grande
    [16.5, 54.16],  # 2. Esquina frontal der del área grande
    [ 0.0, 13.84],  # 3. Línea de meta / área grande (izq)
    [ 0.0, 54.16],  # 4. Línea de meta / área grande (der)
    [ 5.5, 24.84],  # 5. Esquina frontal izq del área chica
    [ 5.5, 43.16],  # 6. Esquina frontal der del área chica
    [11.0, 34.00],  # 7. Punto penal
], dtype=np.float32)

# DLT con normalización de Hartley (estimarH arma Q con obtenerQ2d y resuelve con SVD)
H = estimarH(imgPts.astype(np.float64), objPts.astype(np.float64))

# ------------------------------------------------------------
# Posición de los pies de Enzo en la cancha
# ------------------------------------------------------------
u = 810   # píxel (u, v) de los pies de Enzo en la imagen
v = 520

# Llevamos el píxel a la cancha con la inversa de H
pt_real = multiplicacionMatricial(inversa(H), np.array([u, v, 1]))

# Pasamos de coordenadas homogéneas a (X, Y) dividiendo por la tercera componente
xReal = pt_real[0] / pt_real[2]
yReal = pt_real[1] / pt_real[2]

# La línea de meta es X = 0, así que la distancia es |X|
dist_linea_meta = abs(xReal)
print(f"Coordenadas reales (X, Y): ({xReal:.2f}, {yReal:.2f})")
print(f"Distancia a la línea de meta: {dist_linea_meta:.2f} metros")

El campo de juego es un plano, así que la relación entre sus puntos
$(U, V, 1)^T$ y los píxeles $(u, v, 1)^T$ de la foto es una homografía $H$:

$$\lambda \begin{bmatrix} u \\ v \\ 1 \end{bmatrix} = H \begin{bmatrix} U \\ V \\ 1 \end{bmatrix}$$

Para este ejercicio no necesitamos $K$ ni descomponer $H$ en $R$ y $t$: alcanza con $H$ y su inversa.

**Procedimiento.**
1. Identificamos 7 puntos de referencia en la imagen (esquinas del área grande
   y del área chica, intersecciones con la línea de meta y punto penal) y los
   emparejamos con sus coordenadas en metros en la cancha.
2. Estimamos $H$ con DLT (sistema $Qh = 0$, mínimos cuadrados homogéneos vía SVD),
   usando normalización de Hartley para mejorar el condicionamiento.
3. Verificamos $H$ reproyectando los puntos de la cancha a la imagen y calculando el RMS.
4. Llevamos los pies de Enzo, $(u, v)$, a la cancha con $H^{-1}$ y dividimos por la
   tercera componente para volver de coordenadas homogéneas:
   $(X, Y) = (x'/z', \, y'/z')$.
5. Como la línea del arco es $X = 0$, la distancia pedida es $|X|$.

**Resultado.** Los pies de Enzo, en el píxel $(u, v) = (810, 520)$, se ubican en la
cancha en $(X, Y) \approx (19{,}16;\ 45{,}98)$ m, por lo que patea a aproximadamente
**19,2 m de la línea del arco**.

b) Calcular la homografía de la changa de fútbol de la foto $\texttt{posibleoffside.jpg}$. 
    Trazar sobre la fotografía una recta sobre el pié del jugador del equipo defensor que defina si hay fuera de juego.
    

In [ ]:
# b) Ubicación de los jugadores argentinos y círculos de 1 m
# ============================================================

# ------------------------------------------------------------
# Píxeles (u, v, 1) de los pies de los jugadores argentinos
# ------------------------------------------------------------
# Cada columna es un jugador, leído a ojo sobre la imagen con la grilla.
# La 5ta columna (810, 520) son los pies de Enzo, el mismo píxel del inciso a).
Pimg = np.array([[100, 225, 380, 380, 810, 850],   # u
                 [545, 400, 325, 300, 520, 350],   # v
                 [  1,   1,   1,   1,   1,   1]], dtype=np.float64)

# ------------------------------------------------------------
# Llevamos los jugadores del plano de la imagen a la cancha
# ------------------------------------------------------------
# (X, Y, 1) ~ H^-1 * (u, v, 1). Usamos la H estimada en el inciso a).
PtsReales = multiplicacionMatricial(inversa(H), Pimg)

# Normalizamos las coordenadas homogéneas dividiendo por la tercera fila
PtsReales_norm = PtsReales / PtsReales[2, :]

# Posición de cada jugador sobre la cancha, en metros
X_reales = PtsReales_norm[0, :]
Y_reales = PtsReales_norm[1, :]

# ------------------------------------------------------------
# Pensamos un círculo de 1 m de radio, definido en coordenadas de la cancha

radio = 1.0
y_circ = np.arange(-100, 101) / 100 * radio
x_circ = np.sqrt(radio**2 - y_circ**2)

X_circ = np.hstack([x_circ, -x_circ[::-1]])
Y_circ = np.hstack([y_circ,  y_circ[::-1]])

# ------------------------------------------------------------
# Proyección de los círculos a la imagen con H
# ------------------------------------------------------------
plt.figure(figsize=(12, 8))
plt.imshow(img_rgb)

for i in range(6):
    # Centramos el círculo en la posición real del jugador i: (X, Y, 1)
    CirculoReal_i = np.vstack([X_reales[i] + X_circ,
                               Y_reales[i] + Y_circ,
                               np.ones(X_circ.shape[0])])

    # Proyectamos a la imagen: (u, v, 1) ~ H * (X, Y, 1), y normalizamos
    CirculoImg_i = multiplicacionMatricial(H, CirculoReal_i)
    CirculoImg_i = CirculoImg_i / CirculoImg_i[2, :]

    plt.plot(CirculoImg_i[0, :], CirculoImg_i[1, :], color='black', linewidth=2)

u_jugadores = Pimg[0, :]
v_jugadores = Pimg[1, :]
plt.plot(u_jugadores, v_jugadores, color='black', linestyle='-', linewidth=2, alpha=0.7)

plt.title("Ubicación espacial y círculos tácticos de 1m")
plt.show()

Ubicación de los jugadores argentinos

**Idea.** Ubicamos a cada jugador en la cancha y dibujamos, sobre el piso, un
círculo de 1 m de radio centrado en sus pies. Como todo ocurre sobre el plano de
la cancha, usamos la misma homografía $H$ del inciso a).

**Procedimiento.**
1. Marcamos en la imagen los píxeles $(u, v)$ de los pies de los 6 jugadores argentinos.
2. Los llevamos a la cancha con $H^{-1}$ y normalizamos por la tercera componente,
   obteniendo $(X_i, Y_i)$ en metros.
3. Definimos en coordenadas de la cancha un círculo de radio $r = 1$ m centrado
   en cada jugador: $(X_i + x, \; Y_i + y)$ con $x^2 + y^2 = r^2$.
4. Proyectamos esos puntos a la imagen con $H$ y los dibujamos. Como la
   proyección no conserva las circunferencias, en la imagen se ven como elipses.

c) Trazar una recta paralela a la línea de fondo de la cancha que to que el pie del
último defensor (es decir, el penúltimo jugador adversario, contando al arquero)
para determinar si los jugadores de blanco estaban en fuera de juego. La recta se
define en coordenadas de la cancha y se proyecta a la imagen mediante H.

In [ ]:
# Fuera de juego: recta paralela a la línea de fondo
# ============================================================

img = cv2.imread(r'Path de imagen fuera de juego')
img_rgb = img[:, :, ::-1] 

# Imagen con grilla para leer a ojo las coordenadas (u, v)
plt.figure(figsize=(12, 8))
plt.imshow(img_rgb)
plt.grid(True, color='red', linestyle='--', linewidth=0.5)
plt.xlabel('u (píxeles horizontal)')
plt.ylabel('v (píxeles vertical)')
plt.title('Identificación de píxeles (u, v)')
plt.show()

# ------------------------------------------------------------

# Origen (0, 0) en la esquina inferior izquierda; el arco de esta foto está en X = 105.
objPts2 = np.array([
    [ 88.5, 54.16],  # P1: esquina superior izq del área grande
    [ 88.5, 13.84],  # P2: esquina inferior izq del área grande
    [ 99.5, 43.16],  # P3: esquina superior izq del área chica
    [ 99.5, 24.84],  # P4: esquina inferior izq del área chica
    [ 94.0, 34.00],  # P5: punto penal
    [105.0, 43.16],  # P6: línea superior del área chica con la línea de meta
], dtype=np.float32)

# Píxeles (u, v) de los mismos puntos, leídos de la grilla (mismo orden que objPts2)
imgPts2 = np.array([
    [ 735, 240],  # P1
    [ 730, 760],  # P2
    [1110, 310],  # P3
    [1180, 520],  # P4
    [ 980, 465],  # P5
    [1385, 185],  # P6
], dtype=np.float32)

# ------------------------------------------------------------
# Homografía de esta foto
# DLT con normalización de Hartley. La llamamos H_fuera para no pisar la H de 4a)
H_fuera = estimarH(imgPts2.astype(np.float64), objPts2.astype(np.float64))

# ------------------------------------------------------------
# Posición del pie del último defensor en la cancha
u_def = 825   # píxel (u, v) del pie del último defensor
v_def = 315

# Llevamos el píxel a la cancha con la inversa de H y deshomogeneizamos
pt_real = multiplicacionMatricial(inversa(H_fuera), np.array([u_def, v_def, 1]))
xReal = pt_real[0] / pt_real[2]
yReal = pt_real[1] / pt_real[2]
print(f"Pie del defensor en la cancha (X, Y): ({xReal:.2f}, {yReal:.2f}) m")
print(f"Distancia a la línea de meta: {105.0 - xReal:.2f} m")


# Paralela a la línea de fondo (X = cte) que pasa por el pie del defensor.
# Recorremos todo el ancho de la cancha, Y de 0 a 68 m (101 puntos).
Y_recta = np.arange(0, 101) / 100 * 68.0
X_recta = xReal * np.ones(101)

# Coordenadas homogéneas (X, Y, 1), matriz de 3 x 101
RectaReal = np.vstack([X_recta, Y_recta, np.ones(101)])

# ------------------------------------------------------------
# Proyección de la recta a la imagen con H
# ------------------------------------------------------------
RectaImg = multiplicacionMatricial(H_fuera, RectaReal)
RectaImg = RectaImg / RectaImg[2, :]   # deshomogeneizamos

u_recta = RectaImg[0, :]
v_recta = RectaImg[1, :]

# Nos quedamos solo con los puntos que caen sobre el pasto en el encuadre
v_carteles_top = 250   # altura aproximada de los carteles LED
v_pasto_bottom = 750   # límite inferior del pasto en el encuadre
mask = (v_recta >= v_carteles_top) * (v_recta <= v_pasto_bottom)   # producto = "y" lógico

u_recta_cancha = u_recta[mask]
v_recta_cancha = v_recta[mask]

# ------------------------------------------------------------
plt.figure(figsize=(12, 8))
plt.imshow(img_rgb)
plt.plot(u_recta_cancha, v_recta_cancha, color='red', linewidth=2,
         label=f'Línea de fuera de juego (X = {xReal:.2f} m)')
plt.scatter([u_def], [v_def], color='yellow', s=40, zorder=5)   # pie del defensor
plt.title("Fuera de juego: recta paralela a la línea de fondo")
plt.legend()
plt.show()

Línea de fuera de juego es paralela a la línea de fondo y pasa por el
pie del último defensor (el penúltimo adversario, contando al arquero). En
coordenadas de la cancha es una recta $X = X_{def}$. La proyectamos a la imagen
con la homografía de esta foto.

**Procedimiento.**
1. Identificamos 6 puntos de referencia en la imagen (esquinas del área grande y
   del área chica, punto penal e intersección con la línea de meta) y los
   emparejamos con sus coordenadas en metros según `estadio-azteca.png`.
2. Estimamos $H$ con DLT y normalización de Hartley, y verificamos con el RMS de reproyección.
3. Llevamos el pie del último defensor a la cancha con $H^{-1}$ y normalizamos
   por la tercera componente, obteniendo $X_{def}$.
4. Definimos la recta $\{(X_{def}, Y) : Y \in [0, 68]\}$ en la cancha y la
   proyectamos a la imagen con $H$, recortando la parte que cae sobre el pasto.
5. Comparamos la posición de los jugadores de blanco con la recta.

**Resultado.** El pie del defensor se ubica en $(X, Y) \approx (91{,}2;\ 46{,}6)$ m,
a unos 13,8 m de la línea de meta.

### Ejercicio 5 - Cruzando en rojo - Cálculo de velocidad

a) Calcular los parámetros intrínsecos de la cámara a partir de las diferentes vistas de la esquina y las coordenadas de los puntos en el archivo $\texttt{world\_img\_pts.csv}$

b) A partir de los puntos $\texttt{gs\_10\_09\_point\_correspondences\_v5.txt}$ calcular la pose de la cámara y reproyectar los puntos sobre la imagen $\texttt{img\_10\_18\_47\_351.jpg}$.

c) Calcular la velocidad de los dos primeros vehículos (taxi y camioneta) y de la señora de buzo rojo que cruza la calle. Graficar las velocidades en un plot temporal a lo largo de la secuencia.

### Ejercicio 6 - Aterrizaje suave

a) Aplicar POSIT para encontrar la pose ($\mathbf{R}$ y $\mathbf{t}$) de la cámara del Drone a partir de la imagen $\texttt{row\_00248.png}$. La cámara tiene la matriz intrínseca $\mathbf{K}$. 

b) Estimar la matriz de rotación y traslación para ubicar al drone exactamente sobre la letra H y que la cámara la vea vertical (el punto blanco marca la parte superior izquierda)

---